# NQ 1-Minute – Full Test: Barrier Strategy with Gradient Boosting

The complete test battery from the GC notebooks, applied to **NQ (Nasdaq-100 E-mini)** to see whether these data contain
more usable information:

1. **Win models** (long / short): is `+target` reached before `−stop` within `horizon` minutes? (timing / volatility)
2. **Pure direction test:** is the upper or the lower barrier touched first? (direction only, volatility removed)
3. **Trading simulations** with costs, one position at a time, chosen on validation and applied once to the test period,
   compared with *random direction*, *always long* and *always short* on the same entry candles.

**One data file, no separate test CSV:** the last `TEST_MONTHS` months of the file are the **test period**; of the data
before it, the last `VALIDATION_FRACTION` is validation (threshold selection), the rest is training. Labels never reach
across these boundaries. All inputs are computed on the continuous file, so rolling values and the previous session are
available from the first test candle on.

**Inputs:** only Open, Close, High, Low, Ask, Bid, Delta, Delta/Volume, Session Delta, Session Delta %, Max Delta,
Min Delta, Delta Change, Volume – plus time features from the timestamp, RSI / TWAP / VWAP and (optional) the session
structure (previous session, night session, US opening range). The GC results are printed for comparison.

**Output:** the test period with all original columns (timestamp unchanged) plus probabilities and signals, saved next to
the data file; models and log in `models_barrier_NQ/`.

## 0. Configuration

In [1]:
from pathlib import Path

SYMBOL = "NQ"
# File or folder; a missing ".csv" extension is added automatically. A folder with several CSV files is merged.
DATA_PATH = Path(r"E:\Programierung\Daten\NQ\Candelexport_NQ_M1_MERGED_12.12.2019_26.10.2026")
TEST_MONTHS = 12                # last 12 months of the file = test period (never used for training or selection)
MODEL_DIR = Path("models_barrier_NQ")
LOG_PATH = MODEL_DIR / "barrier_log.csv"

# Model inputs: ONLY these CSV columns (+ time features from "Time UTC" if USE_TIME_FEATURES)
INPUT_COLUMNS = [
    "Open", "Close", "High", "Low", "Ask", "Bid", "Delta", "Delta/Volume",
    "Session Delta", "Session Delta %", "Max Delta", "Min Delta", "Delta Change", "Volume",
]
PERCENT_COLUMNS = ["Delta/Volume", "Session Delta %"]   # exported as text "59.15%" -> 0.5915
USE_TIME_FEATURES = True                                 # hour of day + weekday from the timestamp

# Additional indicators (calculated only from the input columns above)
RSI_PERIODS = [7, 14, 21, 50]          # the set that worked best for GC
TWAP_WINDOWS = [15, 60]                # rolling TWAP in minutes (+ session TWAP, reset after every pause)
VWAP_WINDOWS = [15, 60]                # rolling VWAP in minutes (+ session VWAP, reset after every pause)
USE_SESSION_FEATURES = True            # previous session, night session, US opening range

# Session structure (New York time, daylight saving time handled automatically)
SESSION_START_HOUR_ET = 18      # CME Globex trading day: 18:00 -> 17:00 New York time (daily break 17:00-18:00)
US_OPEN_ET = (9, 30)            # US session open in New York time (= 13:30 UTC in summer, 14:30 UTC in winter)
OPENING_RANGE_MINUTES = 30      # opening range = first 30 minutes after the US open
# GC results for comparison (best GC run, direction AUC validation / test)
GC_REFERENCE = {"15min": (0.5228, 0.5200), "60min": (0.5152, 0.5165)}

# Trade setups (horizon in minutes). Two kinds of barriers:
#   mode "vol":   target/stop = sigma x expected move over the horizon
#                 (std of 1-minute close changes over the last VOL_LOOKBACK candles x sqrt(horizon)), in ticks
#                 -> a trade means the same in quiet and in wild markets
#   mode "fixed": target_ticks / stop_ticks as fixed numbers, e.g.
#                 {"name": "15min_fix", "horizon": 15, "mode": "fixed", "target_ticks": 80, "stop_ticks": 80}
SETUPS = [
    {"name": "15min", "horizon": 15, "mode": "vol", "target_sigma": 1.0, "stop_sigma": 1.0},
    {"name": "60min", "horizon": 60, "mode": "vol", "target_sigma": 1.0, "stop_sigma": 1.0},
]
VOL_LOOKBACK = 60               # candles for the volatility estimate (only past data)
MIN_BARRIER_TICKS = 5           # volatility barriers are never smaller than this

# Futures contract and execution
TICK_SIZE = 0.25                # NQ / MNQ: 0.25 index points
TICK_VALUE_USD = 5.0            # NQ: 5 USD per tick and contract (MNQ: 0.50 USD)
COST_PER_TRADE_TICKS = 2.0      # round trip incl. spread/slippage and commission -> adjust to your account
TARGET_FILL_EXTRA_TICKS = 1     # target counts only if the price trades this many ticks beyond it
MAX_GAP_MINUTES = 5             # no trade if the window contains a longer pause (session break, weekend)
WARMUP_CANDLES = 60             # first candles of a part are not traded (longest rolling input: 60 minutes)

# Data split and training
VALIDATION_FRACTION = 0.2       # last 20 % of the data before the test period -> threshold selection
TRAIN_ROW_STRIDE = 1            # 1 = every candle; e.g. 3 = every 3rd candle (faster, slightly less data)
GBM_PARAMS = dict(
    max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=500,
    l2_regularization=1.0, random_state=42,
)

# Trading rule: trade if max(P_long, P_short) >= threshold; the threshold is chosen on validation (best t-value)
PROB_THRESHOLDS = [round(0.30 + 0.025 * i, 3) for i in range(21)]   # 0.30 ... 0.80
MIN_VALIDATION_TRADES = 100     # thresholds with fewer validation trades are not eligible
BASELINE_RANDOM_RUNS = 5
COMPUTE_FEATURE_IMPORTANCE = True   # permutation importance on a validation sample (a few minutes)

# Direction strategy (section 8.2): direction from the direction model, timing from the win models.
# All combinations are evaluated on validation; the one with the best t-value is applied ONCE to the test data.
DIRECTION_QUANTILES = [0.05, 0.10, 0.20]   # trade the lowest / highest X % of P(up first) -> short / long
TIMING_QUANTILES = [0.0, 0.5, 0.75]        # also require P(win) of the traded side >= this validation quantile (0 = off)
DIRECTION_SIDES = ["both", "short", "long"]

MODEL_DIR.mkdir(parents=True, exist_ok=True)


def resolve_data_files(path: Path) -> list[Path]:
    """File, file without .csv extension, or folder with CSV files."""
    if path.is_file():
        return [path]
    with_ext = Path(str(path) + ".csv")       # note: with_suffix() would cut at the ".2026" in the name
    if with_ext.is_file():
        return [with_ext]
    if path.is_dir():
        files = sorted(path.glob("*.csv"))
        if files:
            return files
    raise FileNotFoundError(f"No CSV found at {path} (also tried {with_ext} and *.csv inside a folder)")


DATA_FILES = resolve_data_files(DATA_PATH)
OUTPUT_CSV_PATH = DATA_FILES[0].with_name(DATA_FILES[0].stem + "_test_signals.csv")
print("Data files:", *[str(f) for f in DATA_FILES], sep="\n  ")
print(f"Output CSV: {OUTPUT_CSV_PATH}")
for s in SETUPS:
    if s.get("mode", "fixed") == "vol":
        print(f"Setup {s['name']}: target {s['target_sigma']} σ / stop {s['stop_sigma']} σ of the expected "
              f"{s['horizon']}-minute move (ticks are shown after the labels are computed)")
    else:
        be = (s["stop_ticks"] + COST_PER_TRADE_TICKS) / (s["target_ticks"] + s["stop_ticks"])
        print(f"Setup {s['name']}: +{s['target_ticks']} / -{s['stop_ticks']} ticks within {s['horizon']} min "
              f"-> break-even win rate ≈ {100 * be:.1f} % (ignoring time exits)")

Data files:
  E:\Programierung\Daten\NQ\Candelexport_NQ_M1_MERGED_12.12.2019_26.10.2026.csv
Output CSV: E:\Programierung\Daten\NQ\Candelexport_NQ_M1_MERGED_12.12.2019_26.10.2026_test_signals.csv
Setup 15min: target 1.0 σ / stop 1.0 σ of the expected 15-minute move (ticks are shown after the labels are computed)
Setup 60min: target 1.0 σ / stop 1.0 σ of the expected 60-minute move (ticks are shown after the labels are computed)


## 1. Imports

In [2]:
import json
import time
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

try:
    from IPython.display import display
except ImportError:  # outside Jupyter
    display = print

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 250)

## 2. Load Data and Split off the Test Period

In [3]:
def load_csv(csv_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Returns (data with input columns + timestamp + source row, original file as text)."""
    text = pd.read_csv(csv_path, sep=";", dtype=str, encoding="utf-8-sig", keep_default_na=False)
    missing = [c for c in ["Time UTC"] + INPUT_COLUMNS if c not in text.columns]
    if missing:
        raise ValueError(f"{csv_path.name}: missing columns {missing}")
    df = pd.DataFrame({"_file": csv_path.name, "_row": np.arange(len(text))})
    for col in INPUT_COLUMNS:
        values = text[col].astype(str).str.strip()
        if col in PERCENT_COLUMNS:
            values = values.str.replace("%", "", regex=False).str.replace("'", "", regex=False).str.replace(" ", "", regex=False)
            values = values.where(values.str.contains(".", regex=False), values.str.replace(",", ".", regex=False))
            num = pd.to_numeric(values, errors="coerce") / 100
            n_bad = int((num.isna() & (values != "") & (values.str.lower() != "nan")).sum())
            if n_bad:
                print(f"⚠ {csv_path.name} {col}: {n_bad:,} values not parseable -> 0")
            df[col] = num.fillna(0.0)
        else:
            df[col] = pd.to_numeric(values, errors="coerce")
    ts = pd.to_datetime(text["Time UTC"], format="%d.%m.%Y %H:%M:%S", utc=True, errors="coerce")
    if ts.isna().mean() > 0.01:                    # other date format -> flexible parsing
        ts = pd.to_datetime(text["Time UTC"], dayfirst=True, utc=True, errors="coerce")
    df["timestamp"] = ts
    return df, text


parts, texts = [], {}
for f in DATA_FILES:
    d, t = load_csv(f)
    parts.append(d)
    texts[f.name] = t
    print(f"{f.name}: {len(d):,} rows")
df_full = pd.concat(parts, ignore_index=True)

# Clean: valid timestamps and OHLC, chronological order, one candle per minute (merged files can overlap)
n0 = len(df_full)
df_full = df_full.dropna(subset=["timestamp", "Open", "High", "Low", "Close"])
df_full = df_full.sort_values(["timestamp"], kind="stable")
dupes = int(df_full["timestamp"].duplicated(keep="last").sum())
df_full = df_full.drop_duplicates("timestamp", keep="last").reset_index(drop=True)
for col in INPUT_COLUMNS:
    df_full[col] = df_full[col].fillna(0.0).astype("float64")
print(f"Cleaned: {len(df_full):,} candles ({n0 - len(df_full) - dupes:,} invalid rows, {dupes:,} duplicate timestamps removed), "
      f"{df_full['timestamp'].min()} → {df_full['timestamp'].max()}")

# Test period = last TEST_MONTHS months
test_start = df_full["timestamp"].max() - pd.DateOffset(months=TEST_MONTHS)
n_before_test = int((df_full["timestamp"] < test_start).sum())
df_train = df_full.iloc[:n_before_test].reset_index(drop=True)
df_test = df_full.iloc[n_before_test:].reset_index(drop=True)
print(f"Training + validation: {df_train['timestamp'].min()} → {df_train['timestamp'].max()} ({len(df_train):,} candles)")
print(f"Test period:           {df_test['timestamp'].min()} → {df_test['timestamp'].max()} ({len(df_test):,} candles)")

Candelexport_NQ_M1_MERGED_12.12.2019_26.10.2026.csv: 2,424,101 rows
Cleaned: 2,424,101 candles (0 invalid rows, 0 duplicate timestamps removed), 2019-12-12 23:00:00+00:00 → 2026-10-26 12:13:00+00:00
Training + validation: 2019-12-12 23:00:00+00:00 → 2025-10-24 20:59:00+00:00 (2,070,502 candles)
Test period:           2025-10-26 22:00:00+00:00 → 2026-10-26 12:13:00+00:00 (353,599 candles)


## 3. Inputs (features)

Only backward-looking values of the input columns: candle shape, price changes, volatility and position in the recent range
(all in ticks, independent of the price level), plus order flow (Ask/Bid, Delta, Session Delta, Max/Min Delta, Volume)
as raw values, ratios and rolling sums. Gradient boosting handles missing values itself.

**Indicators:** RSI for every period in `RSI_PERIODS` (level + change over 5 minutes), **TWAP** and **VWAP** – each as
session value (reset at every session start / after a pause) and as rolling value over `TWAP_WINDOWS` / `VWAP_WINDOWS`.
TWAP/VWAP enter as **distance of the close in ticks** (and relative to the recent candle range), so the model does not
see absolute price levels. Typical price = (High + Low + Close) / 3.

**Session structure** (trading day = 18:00 → 17:00 New York time, like CME Globex):
* **Previous session:** high, low, close and VWAP → distance of the current close in ticks, position inside the previous
  range, previous range; Session Delta at the end of the previous session, its delta sum and delta per volume.
* **Night session (Asia/London):** from the session start until the US open – range so far, position inside it, distance
  to night high/low; after the US open the final night range is used. Flag whether the candle is in the night session.
* **US opening range:** high/low of the first `OPENING_RANGE_MINUTES` after the US open; available only **after** it is
  complete (no look-ahead): range, distance to OR high/low, +1 above / −1 below / 0 inside. Minutes since the US open.

All values use only completed data up to the current candle.

In [4]:
def rsi(close: pd.Series, period: int) -> pd.Series:
    """Relative Strength Index with Wilder smoothing (0..100)."""
    diff = close.diff()
    avg_gain = diff.clip(lower=0).ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss = (-diff.clip(upper=0)).ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    value = 100 - 100 / (1 + avg_gain / avg_loss.replace(0, np.nan))
    value[(avg_loss == 0) & (avg_gain > 0)] = 100.0
    value[(avg_loss == 0) & (avg_gain == 0)] = 50.0
    return value


def session_structure_features(df: pd.DataFrame) -> dict:
    """Previous session, night session and US opening range (New York time, DST-aware)."""
    t = TICK_SIZE
    c, h, l, vol = df["Close"], df["High"], df["Low"], df["Volume"]
    ts_et = df["timestamp"].dt.tz_convert("America/New_York")
    trade_day = (ts_et + pd.Timedelta(hours=24 - SESSION_START_HOUR_ET)).dt.date   # 18:00 ET -> next trading day
    tod = ts_et.dt.hour * 60 + ts_et.dt.minute
    us_open = US_OPEN_ET[0] * 60 + US_OPEN_ET[1]
    same_calendar_day = ts_et.dt.date == trade_day
    f = {}

    # ---- previous session (completed trading day before the current one)
    tp = (h + l + c) / 3
    g = pd.DataFrame({"day": trade_day, "h": h, "l": l, "c": c, "pv": tp * vol, "v": vol,
                      "sd": df["Session Delta"], "d": df["Delta"]})
    daily = g.groupby("day", sort=True).agg(high=("h", "max"), low=("l", "min"), close=("c", "last"),
                                            pv=("pv", "sum"), v=("v", "sum"), sd_end=("sd", "last"),
                                            delta_sum=("d", "sum"))
    daily["vwap"] = daily["pv"] / daily["v"].replace(0, np.nan)
    prev = daily.shift(1).reindex(trade_day.to_numpy())
    prev.index = df.index
    prev_range = (prev["high"] - prev["low"]).replace(0, np.nan)
    f["dist_prev_high_ticks"] = (c - prev["high"]) / t
    f["dist_prev_low_ticks"] = (c - prev["low"]) / t
    f["dist_prev_close_ticks"] = (c - prev["close"]) / t
    f["dist_prev_vwap_ticks"] = (c - prev["vwap"]) / t
    f["prev_range_ticks"] = prev_range / t
    f["pos_in_prev_range"] = (c - prev["low"]) / prev_range
    f["prev_session_delta"] = prev["sd_end"]
    f["prev_delta_sum"] = prev["delta_sum"]
    f["prev_delta_per_volume"] = prev["delta_sum"] / prev["v"].replace(0, np.nan)

    # ---- night session: session start until the US open (running high/low, fixed after the open)
    is_night = (~same_calendar_day) | (tod < us_open)
    night_high = h.where(is_night).groupby(trade_day).cummax().groupby(trade_day).ffill()
    night_low = l.where(is_night).groupby(trade_day).cummin().groupby(trade_day).ffill()
    night_range = (night_high - night_low).replace(0, np.nan)
    f["is_night_session"] = is_night.astype("float32")
    f["night_range_ticks"] = night_range / t
    f["pos_in_night_range"] = (c - night_low) / night_range
    f["dist_night_high_ticks"] = (c - night_high) / t
    f["dist_night_low_ticks"] = (c - night_low) / t

    # ---- US opening range: only after it is complete
    in_or = same_calendar_day & (tod >= us_open) & (tod < us_open + OPENING_RANGE_MINUTES)
    or_high = h.where(in_or).groupby(trade_day).transform("max")
    or_low = l.where(in_or).groupby(trade_day).transform("min")
    or_done = same_calendar_day & (tod >= us_open + OPENING_RANGE_MINUTES)
    or_high, or_low = or_high.where(or_done), or_low.where(or_done)
    f["or_range_ticks"] = (or_high - or_low) / t
    f["dist_or_high_ticks"] = (c - or_high) / t
    f["dist_or_low_ticks"] = (c - or_low) / t
    f["or_position"] = np.where(or_high.isna(), np.nan, np.where(c > or_high, 1.0, np.where(c < or_low, -1.0, 0.0)))
    f["minutes_since_us_open"] = np.where(same_calendar_day, tod - us_open, tod - us_open - 1440).astype("float32")
    return f


def session_ids(ts: pd.Series) -> pd.Series:
    """New session after every pause > MAX_GAP_MINUTES (daily break, weekend)."""
    return (ts.diff().dt.total_seconds().div(60).fillna(np.inf) > MAX_GAP_MINUTES).cumsum()


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    o, h, l, c = df["Open"], df["High"], df["Low"], df["Close"]
    vol, ask, bid, delta = df["Volume"], df["Ask"], df["Bid"], df["Delta"]
    t = TICK_SIZE
    f = {}

    # Candle shape and price changes (ticks)
    f["body_ticks"] = (c - o) / t
    f["range_ticks"] = (h - l) / t
    f["upper_wick_ticks"] = (h - np.maximum(o, c)) / t
    f["lower_wick_ticks"] = (np.minimum(o, c) - l) / t
    ret1 = (c - c.shift(1)) / t
    for k in [1, 3, 5, 10, 15, 30, 60]:
        f[f"ret_{k}_ticks"] = (c - c.shift(k)) / t
    for k in [5, 15, 60]:
        hh, ll = h.rolling(k).max(), l.rolling(k).min()
        f[f"range_mean_{k}"] = f["range_ticks"].rolling(k).mean()
        f[f"ret_std_{k}"] = ret1.rolling(k).std()
        f[f"pos_in_range_{k}"] = (c - ll) / (hh - ll).replace(0, np.nan)
        f[f"dist_high_{k}_ticks"] = (hh - c) / t
        f[f"dist_low_{k}_ticks"] = (c - ll) / t

    # Order flow
    f["volume"] = vol
    f["log_volume"] = np.log1p(vol.clip(lower=0))
    f["ask"] = ask
    f["bid"] = bid
    f["ask_bid_imbalance"] = (ask - bid) / (ask + bid).replace(0, np.nan)
    f["delta"] = delta
    f["delta_per_volume"] = df["Delta/Volume"]
    f["session_delta"] = df["Session Delta"]
    f["session_delta_pct"] = df["Session Delta %"]
    f["session_delta_change_15"] = df["Session Delta"] - df["Session Delta"].shift(15)
    f["max_delta"] = df["Max Delta"]
    f["min_delta"] = df["Min Delta"]
    f["delta_change"] = df["Delta Change"]
    f["max_delta_rel"] = df["Max Delta"] / vol.replace(0, np.nan)
    f["min_delta_rel"] = df["Min Delta"] / vol.replace(0, np.nan)
    for k in [1, 2, 3, 5]:
        f[f"delta_lag_{k}"] = delta.shift(k)
        f[f"delta_per_volume_lag_{k}"] = df["Delta/Volume"].shift(k)
    for k in [5, 15, 60]:
        f[f"delta_sum_{k}"] = delta.rolling(k).sum()
        f[f"delta_ratio_{k}"] = delta.rolling(k).sum() / vol.rolling(k).sum().replace(0, np.nan)
        f[f"volume_rel_{k}"] = vol / vol.rolling(k).mean().replace(0, np.nan)
        f[f"ask_bid_imbalance_{k}"] = ((ask - bid).rolling(k).sum()
                                       / (ask + bid).rolling(k).sum().replace(0, np.nan))

    # RSI
    for p in RSI_PERIODS:
        r = rsi(c, p)
        f[f"rsi_{p}"] = r
        f[f"rsi_{p}_change_5"] = r - r.shift(5)

    # TWAP / VWAP (distance of the close in ticks and relative to the recent candle range)
    tp = (h + l + c) / 3
    session = session_ids(df["timestamp"])
    range_60 = ((h - l) / t).rolling(60, min_periods=5).mean().replace(0, np.nan)
    twap_session = tp.groupby(session).cumsum() / (tp.groupby(session).cumcount() + 1)
    vol_cum = vol.groupby(session).cumsum()
    vwap_session = ((tp * vol).groupby(session).cumsum() / vol_cum.replace(0, np.nan)).groupby(session).ffill()
    averages = {"twap_session": twap_session, "vwap_session": vwap_session}
    for k in TWAP_WINDOWS:
        averages[f"twap_{k}"] = tp.rolling(k).mean()
    for k in VWAP_WINDOWS:
        averages[f"vwap_{k}"] = (tp * vol).rolling(k).sum() / vol.rolling(k).sum().replace(0, np.nan)
    for name, avg in averages.items():
        f[f"dist_{name}_ticks"] = (c - avg) / t
        f[f"dist_{name}_rel"] = (c - avg) / t / range_60
    f["vwap_minus_twap_session_ticks"] = (vwap_session - twap_session) / t

    # Session structure over several days
    if USE_SESSION_FEATURES:
        f.update(session_structure_features(df))

    # Time of day / weekday from the timestamp (UTC)
    if USE_TIME_FEATURES:
        ts = df["timestamp"]
        minute_of_day = ts.dt.hour * 60 + ts.dt.minute
        f["tod_sin"] = np.sin(2 * np.pi * minute_of_day / 1440)
        f["tod_cos"] = np.cos(2 * np.pi * minute_of_day / 1440)
        f["weekday"] = ts.dt.dayofweek
        f["minutes_since_gap"] = minutes_since_gap(ts)

    out = pd.DataFrame(f).replace([np.inf, -np.inf], np.nan).astype("float32")
    return out


def minutes_since_gap(ts: pd.Series) -> pd.Series:
    """Minutes since the last pause > MAX_GAP_MINUTES (session start / after the weekend)."""
    gap = ts.diff().dt.total_seconds().div(60).fillna(np.inf) > MAX_GAP_MINUTES
    block = gap.cumsum()
    return ts.groupby(block).cumcount().astype("float32")


t0 = time.time()
X_full = build_features(df_full)             # continuous file: no warm-up gap at the start of the test period
X_train_all = X_full.iloc[:n_before_test].reset_index(drop=True)
X_test = X_full.iloc[n_before_test:].reset_index(drop=True)
del X_full
FEATURE_COLUMNS = list(X_train_all.columns)
print(f"{len(FEATURE_COLUMNS)} inputs built in {time.time() - t0:.0f} s")
SESSION_PREFIXES = ("rsi_", "dist_twap", "dist_vwap", "vwap_minus", "dist_prev", "prev_", "pos_in_prev", "is_night",
                    "night_", "pos_in_night", "dist_night", "or_", "dist_or", "minutes_since_us_open")
print("Indicator and session inputs:", [c for c in FEATURE_COLUMNS if c.startswith(SESSION_PREFIXES)])

105 inputs built in 6 s
Indicator and session inputs: ['rsi_7', 'rsi_7_change_5', 'rsi_14', 'rsi_14_change_5', 'rsi_21', 'rsi_21_change_5', 'rsi_50', 'rsi_50_change_5', 'dist_twap_session_ticks', 'dist_twap_session_rel', 'dist_vwap_session_ticks', 'dist_vwap_session_rel', 'dist_twap_15_ticks', 'dist_twap_15_rel', 'dist_twap_60_ticks', 'dist_twap_60_rel', 'dist_vwap_15_ticks', 'dist_vwap_15_rel', 'dist_vwap_60_ticks', 'dist_vwap_60_rel', 'vwap_minus_twap_session_ticks', 'dist_prev_high_ticks', 'dist_prev_low_ticks', 'dist_prev_close_ticks', 'dist_prev_vwap_ticks', 'prev_range_ticks', 'pos_in_prev_range', 'prev_session_delta', 'prev_delta_sum', 'prev_delta_per_volume', 'is_night_session', 'night_range_ticks', 'pos_in_night_range', 'dist_night_high_ticks', 'dist_night_low_ticks', 'or_range_ticks', 'dist_or_high_ticks', 'dist_or_low_ticks', 'or_position', 'minutes_since_us_open']


## 4. Labels: Barrier Outcomes for Every Candle

For every candle, a long and a short trade entered at its close are played through minute by minute:
target (+`TARGET_FILL_EXTRA_TICKS` beyond it) → win, stop → loss (stop wins if both are touched in the same candle),
neither → closed at the close after `horizon` minutes. Windows that contain a pause are invalid (not traded, not trained).

With `mode: "vol"` target and stop are set per candle from the volatility of the **past** `VOL_LOOKBACK` candles.

**Direction label (for the pure direction test):** is the upper barrier (+target) touched **before** the lower one
(−target)? Only windows where exactly one side is reached first count; time-outs and candles touching both are excluded.
This removes the volatility question – what remains is only "up or down".

In [5]:
EXIT_TARGET, EXIT_STOP, EXIT_TIME = 1, 2, 3


def barrier_ticks(df: pd.DataFrame, setup: dict):
    """Target and stop in ticks per candle (arrays)."""
    n = len(df)
    if setup.get("mode", "fixed") == "fixed":
        return np.full(n, float(setup["target_ticks"])), np.full(n, float(setup["stop_ticks"]))
    ret1 = df["Close"].diff() / TICK_SIZE
    expected_move = (ret1.rolling(VOL_LOOKBACK, min_periods=VOL_LOOKBACK // 2).std() * np.sqrt(setup["horizon"])).to_numpy()
    target = np.maximum(np.round(setup["target_sigma"] * expected_move), MIN_BARRIER_TICKS)
    stop = np.maximum(np.round(setup["stop_sigma"] * expected_move), MIN_BARRIER_TICKS)
    return target, stop


def barrier_outcomes(df: pd.DataFrame, setup: dict) -> dict:
    horizon = setup["horizon"]
    target_ticks, stop_ticks = barrier_ticks(df, setup)
    c = df["Close"].to_numpy("float64")
    h = df["High"].to_numpy("float64")
    l = df["Low"].to_numpy("float64")
    n = len(df)
    extra = TARGET_FILL_EXTRA_TICKS * TICK_SIZE
    out = {"target_ticks": target_ticks, "stop_ticks": stop_ticks}
    for side, d in (("long", 1), ("short", -1)):
        target = c + d * (target_ticks * TICK_SIZE + extra)
        stop = c - d * stop_ticks * TICK_SIZE
        result = np.full(n, np.nan)
        exit_type = np.zeros(n, dtype="int8")
        minutes = np.zeros(n, dtype="int16")
        done = np.zeros(n, dtype=bool)
        for k in range(1, horizon + 1):
            hk = np.full(n, np.nan)
            lk = np.full(n, np.nan)
            hk[:n - k], lk[:n - k] = h[k:], l[k:]
            if d == 1:
                stop_hit, tgt_hit = lk <= stop, hk >= target
            else:
                stop_hit, tgt_hit = hk >= stop, lk <= target
            stop_hit &= ~done
            tgt_hit &= ~done & ~stop_hit
            result[stop_hit], exit_type[stop_hit], minutes[stop_hit] = -stop_ticks[stop_hit], EXIT_STOP, k
            result[tgt_hit], exit_type[tgt_hit], minutes[tgt_hit] = target_ticks[tgt_hit], EXIT_TARGET, k
            done |= stop_hit | tgt_hit
        ck = np.full(n, np.nan)
        ck[:n - horizon] = c[horizon:]
        open_ = ~done
        result[open_] = d * (ck[open_] - c[open_]) / TICK_SIZE
        exit_type[open_], minutes[open_] = EXIT_TIME, horizon
        out[side] = {"result": result, "exit": exit_type, "minutes": minutes, "win": exit_type == EXIT_TARGET}

    # Direction label: +target before -target (symmetric, no extra tick); both in one candle / time-out -> unresolved
    up_level, down_level = c + target_ticks * TICK_SIZE, c - target_ticks * TICK_SIZE
    up_first = np.zeros(n, dtype=bool)
    resolved = np.zeros(n, dtype=bool)
    decided = np.zeros(n, dtype=bool)
    for k in range(1, horizon + 1):
        hk = np.full(n, np.nan)
        lk = np.full(n, np.nan)
        hk[:n - k], lk[:n - k] = h[k:], l[k:]
        up, down = (hk >= up_level) & ~decided, (lk <= down_level) & ~decided
        only_up, only_down = up & ~down, down & ~up
        up_first |= only_up
        resolved |= only_up | only_down
        decided |= up | down
    out["direction"] = {"up_first": up_first, "resolved": resolved}

    # Valid = complete window inside the file, no pause inside the window, warm-up done
    gap = (df["timestamp"].diff().dt.total_seconds().div(60).fillna(0).to_numpy() > MAX_GAP_MINUTES).astype(int)
    cum = np.cumsum(gap)
    idx = np.arange(n)
    end = np.minimum(idx + horizon, n - 1)
    out["valid"] = ((idx + horizon < n) & ((cum[end] - cum[idx]) == 0) & (idx >= WARMUP_CANDLES)
                    & np.isfinite(target_ticks) & np.isfinite(stop_ticks))
    return out


t0 = time.time()
outcomes_train = {s["name"]: barrier_outcomes(df_train, s) for s in SETUPS}
outcomes_test = {s["name"]: barrier_outcomes(df_test, s) for s in SETUPS}
print(f"Barrier outcomes computed in {time.time() - t0:.0f} s")
for s in SETUPS:
    o = outcomes_train[s["name"]]
    v = o["valid"]
    print(f"[{s['name']}] valid candles {v.sum():,} | win rate if ALWAYS traded: "
          f"long {100 * o['long']['win'][v].mean():.1f} %, short {100 * o['short']['win'][v].mean():.1f} % "
          f"| Ø result long {np.nanmean(o['long']['result'][v]):+.2f}, short {np.nanmean(o['short']['result'][v]):+.2f} ticks (gross)")
    for label, oo in (("training data", o), ("test data", outcomes_test[s["name"]])):
        vv = oo["valid"]
        print(f"      {label}: Ø target {np.mean(oo['target_ticks'][vv]):.1f} / Ø stop {np.mean(oo['stop_ticks'][vv]):.1f} ticks, "
              f"direction resolved in {100 * oo['direction']['resolved'][vv].mean():.1f} % of the windows, "
              f"up first {100 * oo['direction']['up_first'][vv & oo['direction']['resolved']].mean():.1f} %")

Barrier outcomes computed in 4 s
in] valid candles 2,041,216 | win rate if ALWAYS traded: long 27.5 %, short 28.2 % | Ø result long -0.05, short -0.64 ticks (gross)
      training data: Ø target 67.8 / Ø stop 67.8 ticks, direction resolved in 57.7 % of the windows, up first 49.5 %
      test data: Ø target 120.7 / Ø stop 120.7 ticks, direction resolved in 56.6 % of the windows, up first 48.5 %
in] valid candles 1,966,334 | win rate if ALWAYS traded: long 29.5 %, short 30.3 % | Ø result long +1.03, short -1.82 ticks (gross)
      training data: Ø target 133.4 / Ø stop 133.4 ticks, direction resolved in 60.8 % of the windows, up first 49.3 %
      test data: Ø target 240.1 / Ø stop 240.1 ticks, direction resolved in 59.2 % of the windows, up first 48.2 %


## 5. Time Split: Training / Validation

In [6]:
n_train_total = len(df_train)
val_start = int(n_train_total * (1 - VALIDATION_FRACTION))
max_horizon = max(s["horizon"] for s in SETUPS)
# Purge: training labels must not look into the validation period
train_rows = np.arange(0, val_start - max_horizon, TRAIN_ROW_STRIDE)
val_rows = np.arange(val_start, n_train_total)
print(f"Training:   {df_train['timestamp'].iloc[train_rows[0]]} → {df_train['timestamp'].iloc[train_rows[-1]]} "
      f"({len(train_rows):,} candles, stride {TRAIN_ROW_STRIDE})")
print(f"Validation: {df_train['timestamp'].iloc[val_rows[0]]} → {df_train['timestamp'].iloc[val_rows[-1]]} ({len(val_rows):,} candles)")
print(f"Test:       {df_test['timestamp'].iloc[0]} → {df_test['timestamp'].iloc[-1]} ({len(df_test):,} candles)")

Training:   2019-12-12 23:00:00+00:00 → 2024-08-23 11:13:00+00:00 (1,656,341 candles, stride 1)
Validation: 2024-08-23 12:14:00+00:00 → 2025-10-24 20:59:00+00:00 (414,101 candles)
Test:       2025-10-26 22:00:00+00:00 → 2026-10-26 12:13:00+00:00 (353,599 candles)


## 6. Train Long and Short Classifier per Setup

In [7]:
models = {}
auc_rows = []
for s in SETUPS:
    name = s["name"]
    o_tr, o_te = outcomes_train[name], outcomes_test[name]
    tr = train_rows[o_tr["valid"][train_rows]]
    va = val_rows[o_tr["valid"][val_rows]]
    te = np.flatnonzero(o_te["valid"])
    models[name] = {}
    for side in ("long", "short"):
        t0 = time.time()
        clf = HistGradientBoostingClassifier(**GBM_PARAMS)
        clf.fit(X_train_all.iloc[tr], o_tr[side]["win"][tr])
        p_val = clf.predict_proba(X_train_all.iloc[va])[:, 1]
        p_te = clf.predict_proba(X_test.iloc[te])[:, 1]
        models[name][side] = clf
        auc_rows.append({
            "Setup": name, "Side": side, "Train candles": len(tr),
            "Base win rate val %": 100 * o_tr[side]["win"][va].mean(),
            "AUC validation": roc_auc_score(o_tr[side]["win"][va], p_val),
            "AUC test": roc_auc_score(o_te[side]["win"][te], p_te),
            "Ø P val": p_val.mean(), "P val 99%": np.percentile(p_val, 99),
            "Fit seconds": time.time() - t0,
        })
        print(f"[{name} {side}] trained in {time.time() - t0:.0f} s – AUC validation {auc_rows[-1]['AUC validation']:.4f}, "
              f"test {auc_rows[-1]['AUC test']:.4f}")

print("\nAUC = how well the model ranks winners above losers: 0.50 = no skill, 0.55 = weak, 0.60+ = useful")
display(pd.DataFrame(auc_rows).round(4))

in long] trained in 25 s – AUC validation 0.6202, test 0.6088
in short] trained in 21 s – AUC validation 0.6109, test 0.5970
in long] trained in 21 s – AUC validation 0.6258, test 0.6123
in short] trained in 22 s – AUC validation 0.6061, test 0.5741

AUC = how well the model ranks winners above losers: 0.50 = no skill, 0.55 = weak, 0.60+ = useful


,Setup,Side,Train candles,Base win rate val %,AUC validation,AUC test,Ø P val,P val 99%,Fit seconds
0,15min,long,1631645,26.7944,0.6202,0.6088,0.2711,0.4741,24.5456
1,15min,short,1631645,28.4364,0.6109,0.5970,0.2754,0.4790,21.0631
2,60min,long,1570533,28.0322,0.6258,0.6123,0.2911,0.5315,20.9430
3,60min,short,1570533,30.3744,0.6061,0.5741,0.2986,0.5378,22.1860


### 6.1 Which Inputs Matter (permutation importance on a validation sample)

In [8]:
if COMPUTE_FEATURE_IMPORTANCE:
    rng = np.random.default_rng(0)
    for s in SETUPS:
        name = s["name"]
        va = val_rows[outcomes_train[name]["valid"][val_rows]]
        sample = rng.choice(va, size=min(50_000, len(va)), replace=False)
        for side in ("long", "short"):
            imp = permutation_importance(
                models[name][side], X_train_all.iloc[sample], outcomes_train[name][side]["win"][sample],
                scoring="roc_auc", n_repeats=3, random_state=0, n_jobs=-1,
            )
            top = (pd.Series(imp.importances_mean, index=FEATURE_COLUMNS).sort_values(ascending=False).head(12))
            print(f"[{name} {side}] AUC loss when an input is shuffled (top 12):")
            print("   " + ", ".join(f"{k} {v:.4f}" for k, v in top.items()))

in long] AUC loss when an input is shuffled (top 12):
   ret_std_60 0.0915, minutes_since_us_open 0.0202, range_mean_60 0.0153, range_mean_15 0.0127, volume_rel_60 0.0093, minutes_since_gap 0.0067, range_mean_5 0.0041, tod_cos 0.0025, is_night_session 0.0025, or_range_ticks 0.0020, tod_sin 0.0020, prev_range_ticks 0.0018
in short] AUC loss when an input is shuffled (top 12):
   ret_std_60 0.0711, minutes_since_us_open 0.0269, range_mean_15 0.0129, range_mean_60 0.0109, minutes_since_gap 0.0055, tod_cos 0.0044, tod_sin 0.0043, volume_rel_60 0.0043, prev_range_ticks 0.0035, range_mean_5 0.0034, dist_low_60_ticks 0.0016, night_range_ticks 0.0015
in long] AUC loss when an input is shuffled (top 12):
   ret_std_60 0.0921, minutes_since_us_open 0.0178, tod_cos 0.0105, tod_sin 0.0077, minutes_since_gap 0.0068, prev_delta_per_volume 0.0062, prev_range_ticks 0.0059, prev_session_delta 0.0051, or_range_ticks 0.0040, range_mean_15 0.0034, range_mean_60 0.0026, night_range_ticks 0.0024
in short] A

### 6.2 Pure Direction Test: "Up or Down First?"

One classifier per setup on the direction label only (volatility removed). **AUC 0.50 = no direction information**,
0.52–0.55 = weak, above 0.55 = interesting. The bucket table shows on the test data whether higher model probabilities
really go along with more "up first" outcomes – that is what a trading signal needs.

In [9]:
direction_models, direction_rows = {}, []
for s in SETUPS:
    name = s["name"]
    o_tr, o_te = outcomes_train[name], outcomes_test[name]
    ok_tr = o_tr["valid"] & o_tr["direction"]["resolved"]
    ok_te = o_te["valid"] & o_te["direction"]["resolved"]
    tr = train_rows[ok_tr[train_rows]]
    va = val_rows[ok_tr[val_rows]]
    te = np.flatnonzero(ok_te)
    t0 = time.time()
    clf = HistGradientBoostingClassifier(**GBM_PARAMS)
    clf.fit(X_train_all.iloc[tr], o_tr["direction"]["up_first"][tr])
    direction_models[name] = clf
    p_va = clf.predict_proba(X_train_all.iloc[va])[:, 1]
    p_te = clf.predict_proba(X_test.iloc[te])[:, 1]
    y_va, y_te = o_tr["direction"]["up_first"][va], o_te["direction"]["up_first"][te]
    direction_rows.append({
        "Setup": name, "Train windows": len(tr), "Up first val %": 100 * y_va.mean(), "Up first test %": 100 * y_te.mean(),
        "AUC direction validation": roc_auc_score(y_va, p_va), "AUC direction test": roc_auc_score(y_te, p_te),
        "Fit seconds": time.time() - t0,
    })
    buckets = pd.qcut(p_te, 10, labels=False, duplicates="drop")
    table = (pd.DataFrame({"bucket": buckets, "p": p_te, "up": y_te})
             .groupby("bucket").agg(windows=("up", "size"), mean_p=("p", "mean"), actual_up_pct=("up", "mean")))
    table["actual_up_pct"] *= 100
    print(f"[{name}] test: model probability (10 buckets, low -> high) vs. actual share 'up first'")
    display(table.round(3))

print("AUC 0.50 = no direction information | 0.52–0.55 = weak | > 0.55 = interesting")
direction_df = pd.DataFrame(direction_rows)
display(direction_df.round(4))

# Comparison with the best GC run
for _, r in direction_df.iterrows():
    ref = GC_REFERENCE.get(r["Setup"])
    if ref is None:
        continue
    print(f"[{r['Setup']}] direction AUC NQ validation {r['AUC direction validation']:.4f} / test {r['AUC direction test']:.4f}"
          f"  |  GC validation {ref[0]:.4f} / test {ref[1]:.4f}"
          f"  ->  test difference {r['AUC direction test'] - ref[1]:+.4f}")

# Which inputs does the DIRECTION model use? (permutation importance on resolved validation windows)
if COMPUTE_FEATURE_IMPORTANCE:
    rng = np.random.default_rng(1)
    indicator_prefixes = SESSION_PREFIXES
    for s in SETUPS:
        name = s["name"]
        o_tr = outcomes_train[name]
        ok = o_tr["valid"] & o_tr["direction"]["resolved"]
        va = val_rows[ok[val_rows]]
        sample = rng.choice(va, size=min(50_000, len(va)), replace=False)
        imp = permutation_importance(
            direction_models[name], X_train_all.iloc[sample], o_tr["direction"]["up_first"][sample],
            scoring="roc_auc", n_repeats=3, random_state=0, n_jobs=-1,
        )
        importance = pd.Series(imp.importances_mean, index=FEATURE_COLUMNS).sort_values(ascending=False)
        print(f"\n[{name} DIRECTION] top 15 inputs (AUC loss when shuffled):")
        print("   " + ", ".join(f"{k} {v:.4f}" for k, v in importance.head(15).items()))
        ind = importance[[c for c in importance.index if c.startswith(indicator_prefixes)]]
        print(f"[{name} DIRECTION] all indicator and session inputs, best first:")
        display(ind.round(5).to_frame("AUC loss").T)

in] test: model probability (10 buckets, low -> high) vs. actual share 'up first'


,windows,mean_p,actual_up_pct
bucket,,,
0,19798,0.405,46.510
1,19798,0.454,47.227
2,19797,0.472,48.012
3,19798,0.484,48.283
4,19798,0.493,48.434
5,19797,0.503,49.012
6,19798,0.512,49.359
7,19797,0.524,48.487
8,19798,0.540,48.747


in] test: model probability (10 buckets, low -> high) vs. actual share 'up first'


,windows,mean_p,actual_up_pct
bucket,,,
0,19997,0.315,48.167
1,19997,0.414,46.377
2,19997,0.448,47.092
3,19997,0.471,47.147
4,19997,0.489,48.182
5,19997,0.506,50.313
6,19997,0.523,48.947
7,19997,0.545,48.517
8,19998,0.579,48.340


AUC 0.50 = no direction information | 0.52–0.55 = weak | > 0.55 = interesting


,Setup,Train windows,Up first val %,Up first test %,AUC direction validation,AUC direction test,Fit seconds
0,15min,944687,48.5778,48.5041,0.5141,0.5117,11.9102
1,60min,960626,48.0492,48.1702,0.5233,0.5067,11.8143


in] direction AUC NQ validation 0.5141 / test 0.5117  |  GC validation 0.5228 / test 0.5200  ->  test difference -0.0083
in] direction AUC NQ validation 0.5233 / test 0.5067  |  GC validation 0.5152 / test 0.5165  ->  test difference -0.0098

in DIRECTION] top 15 inputs (AUC loss when shuffled):
   rsi_50 0.0031, tod_cos 0.0023, or_range_ticks 0.0021, dist_high_60_ticks 0.0019, pos_in_range_60 0.0017, dist_prev_vwap_ticks 0.0016, ret_std_60 0.0014, dist_prev_low_ticks 0.0013, dist_prev_close_ticks 0.0011, dist_prev_high_ticks 0.0009, minutes_since_gap 0.0007, delta_ratio_60 0.0007, dist_or_low_ticks 0.0007, prev_range_ticks 0.0007, dist_night_high_ticks 0.0006
in DIRECTION] all indicator and session inputs, best first:


,rsi_50,or_range_ticks,dist_prev_vwap_ticks,dist_prev_low_ticks,dist_prev_close_ticks,dist_prev_high_ticks,dist_or_low_ticks,prev_range_ticks,dist_night_high_ticks,dist_vwap_60_rel,dist_or_high_ticks,rsi_21,dist_vwap_session_rel,pos_in_night_range,dist_twap_15_rel,dist_vwap_session_ticks,dist_twap_60_rel,vwap_minus_twap_session_ticks,dist_twap_60_ticks,dist_vwap_15_rel,dist_vwap_15_ticks,rsi_14_change_5,dist_twap_15_ticks,or_position,prev_delta_sum,is_night_session,minutes_since_us_open,pos_in_prev_range,rsi_14,rsi_50_change_5,dist_vwap_60_ticks,dist_twap_session_ticks,rsi_7,night_range_ticks,rsi_21_change_5,dist_twap_session_rel,rsi_7_change_5,dist_night_low_ticks,prev_session_delta,prev_delta_per_volume
AUC loss,0.00313,0.00209,0.00159,0.0013,0.00111,0.00093,0.0007,0.00068,0.00062,0.00051,0.0004,0.00038,0.00034,0.00028,0.00025,0.00018,0.00015,0.00015,0.00014,0.00007,0.00003,0.00002,0.00001,0.0,0.0,0.0,-0.00006,-0.00006,-0.00008,-0.00008,-0.00009,-0.00009,-0.00009,-0.0002,-0.00022,-0.00032,-0.00046,-0.00057,-0.00149,-0.00157



in DIRECTION] top 15 inputs (AUC loss when shuffled):
   night_range_ticks 0.0055, dist_prev_vwap_ticks 0.0049, dist_prev_close_ticks 0.0043, dist_prev_high_ticks 0.0042, prev_session_delta 0.0031, dist_night_high_ticks 0.0030, rsi_50 0.0023, minutes_since_us_open 0.0022, minutes_since_gap 0.0021, ret_std_60 0.0020, dist_or_low_ticks 0.0017, prev_range_ticks 0.0016, delta_ratio_60 0.0016, dist_vwap_session_rel 0.0011, tod_cos 0.0010
in DIRECTION] all indicator and session inputs, best first:


,night_range_ticks,dist_prev_vwap_ticks,dist_prev_close_ticks,dist_prev_high_ticks,prev_session_delta,dist_night_high_ticks,rsi_50,minutes_since_us_open,dist_or_low_ticks,prev_range_ticks,dist_vwap_session_rel,dist_twap_60_rel,rsi_21,dist_or_high_ticks,pos_in_night_range,dist_vwap_60_rel,or_range_ticks,rsi_14_change_5,is_night_session,prev_delta_sum,dist_twap_15_ticks,rsi_7,dist_vwap_15_rel,dist_vwap_15_ticks,rsi_50_change_5,dist_twap_15_rel,or_position,rsi_14,dist_vwap_60_ticks,rsi_21_change_5,dist_twap_60_ticks,dist_prev_low_ticks,vwap_minus_twap_session_ticks,rsi_7_change_5,dist_twap_session_rel,prev_delta_per_volume,dist_vwap_session_ticks,dist_twap_session_ticks,pos_in_prev_range,dist_night_low_ticks
AUC loss,0.00554,0.00494,0.00433,0.00424,0.00312,0.00299,0.00235,0.00224,0.00172,0.00159,0.00115,0.00075,0.00027,0.00018,0.00006,0.00005,0.00003,0.00002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,-0.00001,-0.00006,-0.00006,-0.00008,-0.00012,-0.00014,-0.00024,-0.00036,-0.00066,-0.00112,-0.00115,-0.00167,-0.00245


## 7. Trading Simulation and Threshold Selection (validation only)

One position at a time. At every candle: `P_long`, `P_short`; if the larger one is ≥ threshold, a trade in that direction is
opened at the close; its result and duration come from the barrier outcome of that candle. Net = gross − costs.
The threshold with the best **t-value** (Ø net ÷ std × √trades) and at least `MIN_VALIDATION_TRADES` trades is selected.

In [10]:
def simulate(outcome: dict, rows: np.ndarray, direction: np.ndarray) -> pd.DataFrame:
    """rows: candidate candles in time order, direction: +1 / -1 per candidate."""
    trades, next_free = [], -1
    for r, d in zip(rows, direction):
        if r < next_free:
            continue
        side = outcome["long"] if d == 1 else outcome["short"]
        trades.append({"t": int(r), "direction": int(d), "gross_ticks": float(side["result"][r]),
                       "exit": int(side["exit"][r]), "minutes": int(side["minutes"][r])})
        next_free = r + int(side["minutes"][r])
    return pd.DataFrame(trades)


def summarize(trades: pd.DataFrame, timestamps: pd.Series, days: int) -> dict:
    """timestamps: all candles of the data set (trade rows index into it), days: trading days of the period."""
    if trades.empty:
        return {"Trades": 0}
    net = trades["gross_ticks"].to_numpy() - COST_PER_TRADE_TICKS
    equity = np.concatenate([[0.0], np.cumsum(net * TICK_VALUE_USD)])
    drawdown = (np.maximum.accumulate(equity) - equity).max()
    wins, losses = net[net > 0].sum(), -net[net < 0].sum()
    ts = timestamps.iloc[trades["t"].to_numpy()]
    monthly = pd.Series(net, index=ts.dt.strftime("%Y-%m").to_numpy()).groupby(level=0).sum()
    days = max(days, 1)
    is_long = trades["direction"].to_numpy() == 1
    return {
        "Trades": len(trades), "Trades/day": len(trades) / days, "Long %": 100 * is_long.mean(),
        "Target %": 100 * (trades["exit"] == EXIT_TARGET).mean(),
        "Stop %": 100 * (trades["exit"] == EXIT_STOP).mean(),
        "Time exit %": 100 * (trades["exit"] == EXIT_TIME).mean(),
        "Ø net (ticks)": net.mean(),
        "Ø net long": net[is_long].mean() if is_long.any() else np.nan,
        "Ø net short": net[~is_long].mean() if (~is_long).any() else np.nan,
        "Profit factor": wins / losses if losses > 0 else np.inf,
        "t-value": net.mean() / net.std(ddof=1) * np.sqrt(len(net)) if len(net) > 1 and net.std(ddof=1) > 0 else np.nan,
        "Months in profit": f"{int((monthly > 0).sum())}/{len(monthly)}",
        "Net total USD/contract": net.sum() * TICK_VALUE_USD,
        "Max drawdown USD": drawdown,
    }


def signals(p_long: np.ndarray, p_short: np.ndarray, threshold: float):
    best = np.maximum(p_long, p_short)
    mask = best >= threshold
    direction = np.where(p_long >= p_short, 1, -1)
    return mask, direction


probs = {"val": {}, "test": {}}
chosen = {}
for s in SETUPS:
    name = s["name"]
    o_tr = outcomes_train[name]
    va = val_rows[o_tr["valid"][val_rows]]
    p_l = models[name]["long"].predict_proba(X_train_all.iloc[va])[:, 1]
    p_s = models[name]["short"].predict_proba(X_train_all.iloc[va])[:, 1]
    probs["val"][name] = (va, p_l, p_s)

    val_days = df_train["timestamp"].iloc[va].dt.date.nunique()
    rows = []
    for thr in PROB_THRESHOLDS:
        mask, direction = signals(p_l, p_s, thr)
        tr = simulate(o_tr, va[mask], direction[mask])
        rows.append({"Threshold": thr, "Signal candles": int(mask.sum()),
                     **summarize(tr, df_train["timestamp"], val_days)})
    val_table = pd.DataFrame(rows)
    eligible = val_table[val_table["Trades"] >= MIN_VALIDATION_TRADES]
    if eligible.empty or eligible["t-value"].isna().all():
        chosen[name] = None
        print(f"[{name}] no threshold with ≥ {MIN_VALIDATION_TRADES} validation trades – setup not tradable")
    else:
        best = eligible.loc[eligible["t-value"].idxmax()]
        chosen[name] = float(best["Threshold"])
        print(f"[{name}] chosen threshold {chosen[name]:.3f}: validation Ø net {best['Ø net (ticks)']:+.2f} ticks, "
              f"t-value {best['t-value']:.2f}, {int(best['Trades'])} trades")
    print(f"[{name}] validation results per threshold:")
    display(val_table.round(3))

in] chosen threshold 0.450: validation Ø net -0.72 ticks, t-value -0.39, 3563 trades
in] validation results per threshold:


,Threshold,Signal candles,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.300,196087,27521,75.194,43.876,36.103,34.926,28.971,-1.290,-0.912,-1.586,0.961,-2.397,4/15,-177540.0,197860.0
1,0.325,142543,22416,61.246,43.429,37.701,36.532,25.767,-1.322,-0.869,-1.669,0.961,-2.213,4/15,-148125.0,173155.0
2,0.350,96151,17081,46.669,42.404,39.307,38.364,22.329,-1.974,-2.636,-1.486,0.943,-2.844,3/15,-168580.0,190060.0
3,0.375,60840,12168,33.246,41.297,41.494,40.113,18.393,-1.800,-1.710,-1.864,0.951,-2.125,4/15,-109530.0,138610.0
4,0.400,36869,8319,22.730,40.029,43.323,41.952,14.725,-2.475,-3.504,-1.788,0.938,-2.276,5/15,-102955.0,123835.0
5,0.425,21708,5448,14.885,40.382,45.044,43.319,11.637,-2.206,-1.366,-2.775,0.949,-1.556,6/15,-60095.0,70020.0
6,0.450,12655,3563,9.735,41.426,47.067,45.299,7.634,-0.717,-1.475,-0.181,0.984,-0.389,7/15,-12770.0,43495.0
7,0.475,7017,2241,6.123,42.079,48.550,46.809,4.641,-0.917,-4.545,1.718,0.981,-0.394,9/15,-10280.0,35345.0
8,0.500,3445,1303,3.560,42.134,49.117,47.889,2.993,-1.877,-5.053,0.435,0.961,-0.622,6/15,-12230.0,27880.0
9,0.525,1222,562,1.536,46.797,46.797,51.779,1.423,-9.673,-11.304,-8.237,0.811,-2.143,3/15,-27180.0,29960.0


in] chosen threshold 0.525: validation Ø net +8.14 ticks, t-value 1.33, 795 trades
in] validation results per threshold:


,Threshold,Signal candles,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.300,231455,8294,22.661,45.298,36.050,34.543,29.407,0.066,2.609,-2.039,1.001,0.035,7/15,2750.0,70025.0
1,0.325,194265,7413,20.254,46.459,36.841,35.600,27.560,-0.334,1.553,-1.971,0.995,-0.170,9/15,-12365.0,72250.0
2,0.350,158527,6499,17.757,46.869,38.298,36.144,25.558,0.894,1.567,0.300,1.014,0.428,9/15,29040.0,70915.0
3,0.375,124939,5562,15.197,47.860,39.914,37.415,22.672,1.269,1.285,1.254,1.020,0.561,9/15,35300.0,60145.0
4,0.400,93102,4595,12.555,47.878,41.567,38.455,19.978,1.713,0.555,2.777,1.028,0.696,9/15,39355.0,49810.0
5,0.425,65263,3597,9.828,48.568,43.147,39.589,17.264,3.432,2.810,4.019,1.055,1.185,8/15,61725.0,32130.0
6,0.450,43875,2697,7.369,49.462,43.752,40.823,15.425,2.077,1.386,2.753,1.032,0.613,7/15,28010.0,31385.0
7,0.475,28410,1844,5.038,49.837,45.228,40.944,13.829,3.906,-0.274,8.059,1.061,0.939,8/15,36015.0,24355.0
8,0.500,17412,1230,3.361,50.163,45.691,41.951,12.358,2.398,-2.178,7.005,1.036,0.489,9/15,14750.0,15655.0
9,0.525,9805,795,2.172,50.440,48.679,41.258,10.063,8.142,3.292,13.079,1.122,1.326,10/15,32365.0,12485.0


## 8. Test: Model vs. Random Direction vs. Always Long

Applied **once** with the threshold chosen on validation. The comparisons use exactly the same entry candles.

In [11]:
test_days = df_test["timestamp"].dt.date.nunique()
test_rows = []
test_trades = {}
for s in SETUPS:
    name = s["name"]
    o_te = outcomes_test[name]
    te = np.flatnonzero(o_te["valid"])
    p_l = models[name]["long"].predict_proba(X_test.iloc[te])[:, 1]
    p_s = models[name]["short"].predict_proba(X_test.iloc[te])[:, 1]
    probs["test"][name] = (te, p_l, p_s)
    thr = chosen[name]
    if thr is None:
        continue
    mask, direction = signals(p_l, p_s, thr)
    rows_sig = te[mask]
    strategies = {
        "Model": [direction[mask]],
        f"Random direction (Ø {BASELINE_RANDOM_RUNS})": [
            np.random.default_rng(seed).choice([-1, 1], size=mask.sum()) for seed in range(BASELINE_RANDOM_RUNS)
        ],
        "Always long": [np.ones(mask.sum(), dtype=int)],
    }
    for strategy, dirs in strategies.items():
        sums = []
        for d in dirs:
            tr = simulate(o_te, rows_sig, d)
            sums.append(summarize(tr, df_test["timestamp"], test_days))
            if strategy == "Model":
                test_trades[name] = tr
        row = {"Setup": name, "Threshold": thr, "Strategy": strategy, "Signal candles": int(mask.sum())}
        if len(sums) == 1:
            row.update(sums[0])
        else:
            for key in sums[0]:
                vals = [x.get(key) for x in sums]
                row[key] = vals[0] if isinstance(vals[0], str) else float(np.nanmean(vals))
        test_rows.append(row)

test_df = pd.DataFrame(test_rows)
print(f"Rules: costs {COST_PER_TRADE_TICKS} ticks, target fill +{TARGET_FILL_EXTRA_TICKS} tick, stop first in the same "
      f"candle, no trades across pauses > {MAX_GAP_MINUTES} min, tick value {TICK_VALUE_USD} USD")
display(test_df.round(2))

Rules: costs 2.0 ticks, target fill +1 tick, stop first in the same candle, no trades across pauses > 5 min, tick value 5.0 USD


,Setup,Threshold,Strategy,Signal candles,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,15min,0.45,Model,11222,3016.0,9.61,40.32,44.16,45.76,10.08,-2.77,-1.12,-3.88,0.96,-1.06,7/13,-41785.0,69845.0
1,15min,0.45,Random direction (Ø 5),11222,3008.6,9.58,49.79,44.21,45.62,10.17,-3.83,-2.65,-5.01,0.94,-1.47,4/13,-57698.0,87471.0
2,15min,0.45,Always long,11222,3010.0,9.59,100.00,44.42,45.35,10.23,-1.98,-1.98,NaN,0.97,-0.76,4/13,-29805.0,47270.0
3,60min,0.52,Model,11936,775.0,2.47,48.00,41.55,41.81,16.65,-2.60,-1.99,-3.16,0.97,-0.31,5/13,-10080.0,31405.0
4,60min,0.52,Random direction (Ø 5),11936,775.2,2.47,49.74,41.38,42.05,16.56,-2.39,-1.74,-3.04,0.98,-0.28,3/13,-9282.0,47067.0
5,60min,0.52,Always long,11936,776.0,2.47,100.00,42.01,41.62,16.37,-0.98,-0.98,NaN,0.99,-0.12,7/13,-3795.0,31265.0


### 8.1 For Information Only: All Thresholds on the Test Data

Do **not** pick the threshold from this table – that would be fitting to the test year. It only shows whether the
result is stable around the chosen threshold.

In [12]:
for s in SETUPS:
    name = s["name"]
    te, p_l, p_s = probs["test"][name]
    rows = []
    for thr in PROB_THRESHOLDS:
        mask, direction = signals(p_l, p_s, thr)
        tr = simulate(outcomes_test[name], te[mask], direction[mask])
        rows.append({"Threshold": thr, **summarize(tr, df_test["timestamp"], test_days)})
    print(f"[{name}] test results per threshold (chosen on validation: {chosen[name]})")
    display(pd.DataFrame(rows).round(3))

in] test results per threshold (chosen on validation: 0.45)


,Threshold,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.300,22714,72.338,45.413,34.613,34.992,30.395,-1.690,-1.562,-1.796,0.963,-2.269,3/13,-191920.0,262650.0
1,0.325,18337,58.398,44.467,35.938,36.985,27.076,-1.934,-1.784,-2.055,0.959,-2.284,3/13,-177350.0,247245.0
2,0.350,13871,44.175,43.732,37.503,38.851,23.646,-2.257,-2.646,-1.954,0.954,-2.250,2/13,-156500.0,203055.0
3,0.375,9884,31.478,41.977,39.741,39.903,20.356,-0.973,-1.254,-0.770,0.981,-0.787,7/13,-48100.0,105990.0
4,0.400,6695,21.322,40.224,41.583,40.911,17.506,-0.654,0.238,-1.254,0.988,-0.417,6/13,-21885.0,74870.0
5,0.425,4406,14.032,39.537,43.168,43.463,13.368,-0.360,0.892,-1.178,0.994,-0.174,8/13,-7920.0,68180.0
6,0.450,3016,9.605,40.318,44.164,45.756,10.080,-2.771,-1.125,-3.883,0.956,-1.060,7/13,-41785.0,69845.0
7,0.475,2078,6.618,40.375,44.273,48.316,7.411,-5.667,-4.224,-6.643,0.915,-1.778,4/13,-58875.0,78370.0
8,0.500,1322,4.210,37.443,45.991,48.714,5.295,-4.492,-3.917,-4.836,0.932,-1.145,5/13,-29690.0,46205.0
9,0.525,732,2.331,34.563,45.628,49.044,5.328,-4.980,-3.490,-5.766,0.923,-0.980,6/13,-18225.0,24150.0


in] test results per threshold (chosen on validation: 0.525)


,Threshold,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.300,6780,21.592,44.395,35.354,34.130,30.516,-0.372,0.433,-1.014,0.996,-0.141,8/13,-12595.0,82270.0
1,0.325,5983,19.054,44.392,35.634,34.999,29.367,-1.986,-1.983,-1.988,0.978,-0.704,3/13,-59410.0,106670.0
2,0.350,5203,16.570,45.685,35.653,36.441,27.907,-3.689,-3.793,-3.600,0.959,-1.232,2/13,-95960.0,120240.0
3,0.375,4371,13.920,46.168,36.651,37.680,25.669,-4.236,-3.964,-4.469,0.954,-1.266,3/13,-92570.0,130655.0
4,0.400,3593,11.443,46.646,36.989,38.881,24.130,-4.436,-2.033,-6.536,0.952,-1.207,5/13,-79685.0,120260.0
5,0.425,2828,9.006,46.924,38.296,39.215,22.489,-1.126,3.330,-5.066,0.988,-0.270,8/13,-15925.0,52960.0
6,0.450,2180,6.943,47.615,39.404,39.266,21.330,1.377,1.318,1.431,1.015,0.282,8/13,15010.0,55405.0
7,0.475,1581,5.035,47.249,41.682,39.469,18.849,6.846,6.495,7.159,1.073,1.167,10/13,54115.0,37120.0
8,0.500,1123,3.576,46.750,42.565,40.606,16.830,3.822,-1.686,8.657,1.039,0.549,9/13,21460.0,31755.0
9,0.525,775,2.468,48.000,41.548,41.806,16.645,-2.601,-1.992,-3.164,0.974,-0.306,5/13,-10080.0,31405.0


## 8.2 Direction Strategy: Direction Model + Timing

* **Direction:** short if `P(up first)` is in the lowest `X %`, long if it is in the highest `X %` (cut-offs from the
  validation distribution, `DIRECTION_QUANTILES`).
* **Timing:** additionally the win model of the traded side must be at least at its validation quantile
  `TIMING_QUANTILES` (0 = no filter) – i.e. a move large enough to reach the target is expected.
* **Sides:** both, short only, long only (`DIRECTION_SIDES`).
* Same exits as above (volatility target / stop, stop first in the same candle, +1 tick for the target, time exit),
  one position at a time, costs included.

All combinations are evaluated on the **validation period**; the best t-value (with at least `MIN_VALIDATION_TRADES`
trades) is applied **once** to the test data and compared with random direction, always long and always short on exactly
the same entry candles. Long and short are reported separately.

In [13]:
def direction_signals(p_up, p_long, p_short, cfg):
    long_ = (p_up >= cfg["p_up_high"]) & (p_long >= cfg["p_long_min"])
    short_ = (p_up <= cfg["p_up_low"]) & (p_short >= cfg["p_short_min"])
    if cfg["side"] == "short":
        long_[:] = False
    elif cfg["side"] == "long":
        short_[:] = False
    return long_ | short_, np.where(long_, 1, -1)


dir_chosen, dir_test_rows, dir_trades, dir_signal_rows = {}, [], {}, {}
for s in SETUPS:
    name = s["name"]
    o_tr, o_te = outcomes_train[name], outcomes_test[name]
    va = val_rows[o_tr["valid"][val_rows]]
    te = np.flatnonzero(o_te["valid"])
    pu_va = direction_models[name].predict_proba(X_train_all.iloc[va])[:, 1]
    pu_te = direction_models[name].predict_proba(X_test.iloc[te])[:, 1]
    pl_va, ps_va = probs["val"][name][1], probs["val"][name][2]
    pl_te, ps_te = probs["test"][name][1], probs["test"][name][2]
    val_days = df_train["timestamp"].iloc[va].dt.date.nunique()

    # ---- grid on validation
    grid = []
    for q in DIRECTION_QUANTILES:
        for tq in TIMING_QUANTILES:
            for side in DIRECTION_SIDES:
                cfg = {
                    "direction_quantile": q, "timing_quantile": tq, "side": side,
                    "p_up_low": float(np.quantile(pu_va, q)), "p_up_high": float(np.quantile(pu_va, 1 - q)),
                    "p_long_min": float(np.quantile(pl_va, tq)) if tq > 0 else -np.inf,
                    "p_short_min": float(np.quantile(ps_va, tq)) if tq > 0 else -np.inf,
                }
                mask, d = direction_signals(pu_va, pl_va, ps_va, cfg)
                tr = simulate(o_tr, va[mask], d[mask])
                grid.append({**cfg, **summarize(tr, df_train["timestamp"], val_days)})
    grid_df = pd.DataFrame(grid)
    eligible = grid_df[(grid_df["Trades"] >= MIN_VALIDATION_TRADES) & grid_df["t-value"].notna()]
    print(f"[{name}] validation: all combinations")
    display(grid_df.drop(columns=["p_long_min", "p_short_min"]).round(3))
    if eligible.empty:
        print(f"[{name}] no combination with ≥ {MIN_VALIDATION_TRADES} validation trades")
        continue
    best = eligible.loc[eligible["t-value"].idxmax()]
    cfg = {k: best[k] for k in ["direction_quantile", "timing_quantile", "side", "p_up_low", "p_up_high",
                               "p_long_min", "p_short_min"]}
    dir_chosen[name] = cfg
    print(f"[{name}] chosen on validation: side {cfg['side']}, direction {cfg['direction_quantile']:.0%}, "
          f"timing quantile {cfg['timing_quantile']} -> validation Ø net {best['Ø net (ticks)']:+.2f} ticks, "
          f"t-value {best['t-value']:.2f}, {int(best['Trades'])} trades")

    # ---- test, applied once
    mask, d = direction_signals(pu_te, pl_te, ps_te, cfg)
    rows_sig = te[mask]
    dir_signal_rows[name] = (rows_sig, d[mask])
    n_sig = int(mask.sum())
    strategies = {
        "Direction model": [d[mask]],
        f"Random direction (Ø {BASELINE_RANDOM_RUNS})": [
            np.random.default_rng(seed).choice([-1, 1], size=n_sig) for seed in range(BASELINE_RANDOM_RUNS)
        ],
        "Always long": [np.ones(n_sig, dtype=int)],
        "Always short": [-np.ones(n_sig, dtype=int)],
    }
    for strategy, dirs in strategies.items():
        sums = []
        for dd in dirs:
            tr = simulate(o_te, rows_sig, dd)
            sums.append(summarize(tr, df_test["timestamp"], test_days))
            if strategy == "Direction model":
                dir_trades[name] = tr
        row = {"Setup": name, "Side": cfg["side"], "Direction quantile": cfg["direction_quantile"],
               "Timing quantile": cfg["timing_quantile"], "Strategy": strategy, "Signal candles": n_sig}
        if len(sums) == 1:
            row.update(sums[0])
        else:
            for key in sums[0]:
                vals = [x.get(key) for x in sums]
                row[key] = vals[0] if isinstance(vals[0], str) else float(np.nanmean(vals))
        dir_test_rows.append(row)

    # ---- for information only: all combinations on the test data
    info = []
    for _, g in grid_df.iterrows():
        c = {k: g[k] for k in ["direction_quantile", "timing_quantile", "side", "p_up_low", "p_up_high",
                               "p_long_min", "p_short_min"]}
        m2, d2 = direction_signals(pu_te, pl_te, ps_te, c)
        info.append({"direction_quantile": c["direction_quantile"], "timing_quantile": c["timing_quantile"],
                     "side": c["side"], **summarize(simulate(o_te, te[m2], d2[m2]), df_test["timestamp"], test_days)})
    dir_signal_rows[name + "_info"] = pd.DataFrame(info)

print("TEST – direction strategy with the combination chosen on validation:")
dir_test_df = pd.DataFrame(dir_test_rows)
display(dir_test_df.round(2))
for s in SETUPS:
    if s["name"] + "_info" in dir_signal_rows:
        print(f"[{s['name']}] for information only – all combinations on the test data (do not pick from here):")
        display(dir_signal_rows[s["name"] + "_info"].round(3))

in] validation: all combinations


,direction_quantile,timing_quantile,side,p_up_low,p_up_high,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.05,0.00,both,0.421,0.568,7111,19.429,50.809,30.165,27.774,42.062,-0.002,-1.719,1.772,1.000,-0.001,8/15,-55.0,72380.0
1,0.05,0.00,short,0.421,0.568,3505,9.577,0.000,30.642,25.307,44.051,2.425,NaN,2.425,1.057,1.092,7/15,42495.0,44120.0
2,0.05,0.00,long,0.421,0.568,3623,9.899,100.000,29.727,30.003,40.270,-1.423,-1.423,NaN,0.969,-0.630,5/15,-25785.0,64505.0
3,0.05,0.50,both,0.421,0.568,5147,14.063,52.322,34.894,31.941,33.165,1.145,-0.341,2.775,1.025,0.605,7/15,29460.0,44445.0
4,0.05,0.50,short,0.421,0.568,2457,6.713,0.000,36.345,30.077,33.578,3.426,NaN,3.426,1.080,1.283,9/15,42090.0,29505.0
5,0.05,0.50,long,0.421,0.568,2699,7.374,100.000,33.642,33.605,32.753,0.009,0.009,NaN,1.000,0.003,7/15,120.0,45210.0
6,0.05,0.75,both,0.421,0.568,3778,10.322,52.700,37.242,34.648,28.110,-0.442,-2.564,1.922,0.991,-0.201,8/15,-8345.0,37950.0
7,0.05,0.75,short,0.421,0.568,1787,4.883,0.000,38.892,32.513,28.595,1.900,NaN,1.900,1.045,0.637,8/15,16975.0,25815.0
8,0.05,0.75,long,0.421,0.568,1992,5.443,100.000,35.743,36.546,27.711,-2.585,-2.585,NaN,0.950,-0.807,6/15,-25750.0,41170.0
9,0.10,0.00,both,0.443,0.547,13755,37.582,51.770,30.825,28.695,40.480,-0.110,-1.210,1.069,0.997,-0.113,9/15,-7590.0,82035.0


in] chosen on validation: side short, direction 10%, timing quantile 0.5 -> validation Ø net +2.40 ticks, t-value 1.47, 4822 trades
in] validation: all combinations


,direction_quantile,timing_quantile,side,p_up_low,p_up_high,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.05,0.00,both,0.366,0.615,2022,5.525,50.445,30.613,27.399,41.988,2.361,5.563,-0.898,1.025,0.397,7/15,23870.0,45080.0
1,0.05,0.00,short,0.366,0.615,1013,2.768,0.000,29.121,24.679,46.199,-3.188,NaN,-3.188,0.963,-0.416,7/15,-16145.0,49580.0
2,0.05,0.00,long,0.366,0.615,1030,2.814,100.000,31.845,30.485,37.670,6.897,6.897,NaN,1.064,0.756,8/15,35520.0,36175.0
3,0.05,0.50,both,0.366,0.615,1562,4.268,51.857,33.291,29.962,36.748,4.058,6.062,1.900,1.042,0.591,10/15,31695.0,34270.0
4,0.05,0.50,short,0.366,0.615,755,2.063,0.000,32.185,26.623,41.192,1.199,NaN,1.199,1.015,0.134,10/15,4525.0,35835.0
5,0.05,0.50,long,0.366,0.615,812,2.219,100.000,34.236,33.374,32.389,5.310,5.310,NaN,1.049,0.514,8/15,21560.0,33225.0
6,0.05,0.75,both,0.366,0.615,1126,3.077,52.487,36.767,32.416,30.817,6.687,11.230,1.669,1.072,0.836,9/15,37650.0,55245.0
7,0.05,0.75,short,0.366,0.615,537,1.467,0.000,34.637,29.795,35.568,1.097,NaN,1.097,1.013,0.102,9/15,2945.0,24405.0
8,0.05,0.75,long,0.366,0.615,593,1.620,100.000,38.617,35.076,26.307,11.447,11.447,NaN,1.111,0.973,10/14,33940.0,41635.0
9,0.10,0.00,both,0.404,0.576,3813,10.418,52.216,31.681,28.350,39.969,0.626,4.922,-4.069,1.008,0.171,8/15,11930.0,77170.0


in] chosen on validation: side long, direction 10%, timing quantile 0.75 -> validation Ø net +11.35 ticks, t-value 1.58, 1074 trades
TEST – direction strategy with the combination chosen on validation:


,Setup,Side,Direction quantile,Timing quantile,Strategy,Signal candles,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,15min,short,0.1,0.50,Direction model,25556,4639.0,14.77,0.00,33.48,31.21,35.31,0.35,NaN,0.35,1.01,0.20,9/13,8005.0,79085.0
1,15min,short,0.1,0.50,Random direction (Ø 5),25556,4637.4,14.77,50.02,31.60,33.19,35.21,-4.20,-6.96,-1.43,0.92,-2.47,3/13,-97304.0,106138.0
2,15min,short,0.1,0.50,Always long,25556,4635.0,14.76,100.00,30.74,34.15,35.10,-5.10,-5.10,NaN,0.90,-3.00,3/13,-118195.0,120295.0
3,15min,short,0.1,0.50,Always short,25556,4639.0,14.77,0.00,33.48,31.21,35.31,0.35,NaN,0.35,1.01,0.20,9/13,8005.0,79085.0
4,60min,long,0.1,0.75,Direction model,20360,1249.0,3.98,100.00,37.31,37.87,24.82,2.00,2.00,NaN,1.02,0.24,9/13,12510.0,62105.0
5,60min,long,0.1,0.75,Random direction (Ø 5),20360,1247.2,3.97,50.54,37.46,37.73,24.81,-2.88,1.76,-7.61,0.98,-0.34,7/13,-17939.0,67899.0
6,60min,long,0.1,0.75,Always long,20360,1249.0,3.98,100.00,37.31,37.87,24.82,2.00,2.00,NaN,1.02,0.24,9/13,12510.0,62105.0
7,60min,long,0.1,0.75,Always short,20360,1246.0,3.97,0.00,37.48,37.80,24.72,-7.40,NaN,-7.40,0.94,-0.88,4/13,-46105.0,71330.0


in] for information only – all combinations on the test data (do not pick from here):


,direction_quantile,timing_quantile,side,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.05,0.00,both,7764,24.726,51.713,29.598,27.087,43.315,1.783,2.480,1.038,1.033,1.081,8/13,69230.0,90440.0
1,0.05,0.00,short,3764,11.987,0.000,28.905,25.638,45.457,1.237,NaN,1.237,1.026,0.605,7/13,23285.0,55165.0
2,0.05,0.00,long,4028,12.828,100.000,30.288,28.401,41.311,2.506,2.506,NaN,1.041,0.984,8/13,50475.0,46935.0
3,0.05,0.50,both,5763,18.354,53.739,33.732,30.835,35.433,2.417,3.126,1.594,1.043,1.250,8/13,69650.0,95810.0
4,0.05,0.50,short,2675,8.519,0.000,33.757,30.430,35.813,1.832,NaN,1.832,1.038,0.791,9/13,24505.0,56690.0
5,0.05,0.50,long,3104,9.885,100.000,33.827,31.153,35.019,3.290,3.290,NaN,1.052,1.098,8/13,51060.0,46535.0
6,0.05,0.75,both,4244,13.516,53.699,35.721,33.082,31.197,2.818,3.315,2.241,1.050,1.249,8/13,59800.0,61895.0
7,0.05,0.75,short,1970,6.274,0.000,35.888,32.640,31.472,2.379,NaN,2.379,1.052,0.934,8/13,23435.0,33455.0
8,0.05,0.75,long,2283,7.271,100.000,35.611,33.377,31.012,3.372,3.372,NaN,1.052,0.943,8/13,38490.0,34455.0
9,0.10,0.00,both,13380,42.611,52.242,29.761,28.296,41.943,-0.520,-0.118,-0.960,0.990,-0.458,7/13,-34780.0,114895.0


in] for information only – all combinations on the test data (do not pick from here):


,direction_quantile,timing_quantile,side,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.05,0.00,both,2659,8.468,53.291,30.839,31.064,38.097,-8.794,-5.822,-12.184,0.927,-1.565,5/13,-116910.0,137785.0
1,0.05,0.00,short,1252,3.987,0.000,29.952,27.796,42.252,-11.089,NaN,-11.089,0.893,-1.601,5/13,-69415.0,101930.0
2,0.05,0.00,long,1434,4.567,100.000,31.799,33.473,34.728,-3.573,-3.573,NaN,0.974,-0.414,6/13,-25615.0,72960.0
3,0.05,0.50,both,2110,6.720,54.408,33.223,32.986,33.791,-3.100,-4.358,-1.599,0.974,-0.490,7/13,-32705.0,76825.0
4,0.05,0.50,short,965,3.073,0.000,32.850,29.119,38.031,-0.638,NaN,-0.638,0.993,-0.086,6/13,-3080.0,45390.0
5,0.05,0.50,long,1154,3.675,100.000,33.709,35.962,30.329,-3.679,-3.679,NaN,0.973,-0.378,7/13,-21230.0,79915.0
6,0.05,0.75,both,1534,4.885,52.803,34.355,34.485,31.160,-2.022,-1.677,-2.407,0.983,-0.275,9/13,-15505.0,63040.0
7,0.05,0.75,short,725,2.309,0.000,33.103,30.897,36.000,-1.778,NaN,-1.778,0.981,-0.211,6/13,-6445.0,64765.0
8,0.05,0.75,long,812,2.586,100.000,35.468,37.808,26.724,-2.377,-2.377,NaN,0.983,-0.203,8/13,-9650.0,48275.0
9,0.10,0.00,both,4353,13.863,53.159,31.702,30.439,37.859,-2.231,-1.043,-3.579,0.979,-0.564,8/13,-48560.0,94870.0


## 9. Write Test CSV with Signals, Save Models and Log

In [14]:
# Output = test period rows of the original file (all columns as text, timestamp unchanged) + probabilities and signals
test_text = pd.concat(
    [texts[f].iloc[[r]] for f, r in zip(df_test["_file"], df_test["_row"])] if len(texts) > 1
    else [texts[df_test["_file"].iloc[0]].iloc[df_test["_row"].to_numpy()]],
    ignore_index=True,
)
add = pd.DataFrame(index=df_test.index)
for s in SETUPS:
    name = s["name"]
    te, p_l, p_s = probs["test"][name]
    col_l, col_s, col_sig = f"P_long_{name}", f"P_short_{name}", f"Signal_{name}"
    add[col_l] = np.nan
    add[col_s] = np.nan
    add.loc[te, col_l] = np.round(p_l, 4)
    add.loc[te, col_s] = np.round(p_s, 4)
    add[col_sig] = 0
    if chosen[name] is not None:
        mask, direction = signals(p_l, p_s, chosen[name])
        add.loc[te[mask], col_sig] = direction[mask]
    add[f"Target_ticks_{name}"] = outcomes_test[name]["target_ticks"]
    add[f"Stop_ticks_{name}"] = outcomes_test[name]["stop_ticks"]
    add[f"P_up_first_{name}"] = np.nan
    add.loc[te, f"P_up_first_{name}"] = np.round(direction_models[name].predict_proba(X_test.iloc[te])[:, 1], 4)
    add[f"Trade_{name}"] = 0                         # trades actually taken (one position at a time)
    if name in test_trades and not test_trades[name].empty:
        add.loc[test_trades[name]["t"].to_numpy(), f"Trade_{name}"] = test_trades[name]["direction"].to_numpy()
    add[f"DirSignal_{name}"] = 0                     # direction strategy (section 8.2): signal / trades taken
    add[f"DirTrade_{name}"] = 0
    if name in dir_signal_rows:
        rows_sig, dirs = dir_signal_rows[name]
        add.loc[rows_sig, f"DirSignal_{name}"] = dirs
    if name in dir_trades and not dir_trades[name].empty:
        add.loc[dir_trades[name]["t"].to_numpy(), f"DirTrade_{name}"] = dir_trades[name]["direction"].to_numpy()

out = pd.concat([test_text.reset_index(drop=True), add.reset_index(drop=True)], axis=1)
out.to_csv(OUTPUT_CSV_PATH, sep=";", index=False, encoding="utf-8-sig")
print(f"✓ Written: {OUTPUT_CSV_PATH} ({len(out):,} test rows, new columns: {list(add.columns)})")

run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
log_rows = []
for s in SETUPS:
    name = s["name"]
    path = MODEL_DIR / f"barrier_{SYMBOL}_{name}_{run_ts}.joblib"
    joblib.dump({
        "models": models[name], "direction_model": direction_models.get(name),
        "direction_strategy": dir_chosen.get(name),
        "features": FEATURE_COLUMNS, "setup": s, "threshold": chosen[name],
        "vol_lookback": VOL_LOOKBACK, "min_barrier_ticks": MIN_BARRIER_TICKS,
        "trained_until": str(df_train["timestamp"].iloc[val_start - 1]), "test_period_start": str(test_start), "input_columns": INPUT_COLUMNS,
        "use_time_features": USE_TIME_FEATURES, "tick_size": TICK_SIZE,
        "rsi_periods": RSI_PERIODS, "twap_windows": TWAP_WINDOWS, "vwap_windows": VWAP_WINDOWS,
        "use_session_features": USE_SESSION_FEATURES, "session_start_hour_et": SESSION_START_HOUR_ET, "us_open_et": US_OPEN_ET, "opening_range_minutes": OPENING_RANGE_MINUTES,
    }, path)
    joblib.dump(joblib.load(path), MODEL_DIR / f"barrier_{SYMBOL}_{name}_latest.joblib")
    print(f"✓ Saved: {path.resolve()}")
    model_row = test_df[(test_df["Setup"] == name) & (test_df["Strategy"] == "Model")]
    auc = pd.DataFrame(auc_rows)
    log_rows.append({
        "run_timestamp": run_ts, "setup": name, "horizon": s["horizon"], "mode": s.get("mode", "fixed"),
        "target": s.get("target_ticks", s.get("target_sigma")), "stop": s.get("stop_ticks", s.get("stop_sigma")),
        "threshold": chosen[name],
        "auc_direction_val": next((r["AUC direction validation"] for r in direction_rows if r["Setup"] == name), None),
        "auc_direction_test": next((r["AUC direction test"] for r in direction_rows if r["Setup"] == name), None),
        "auc_val_long": float(auc[(auc.Setup == name) & (auc.Side == "long")]["AUC validation"].iloc[0]),
        "auc_val_short": float(auc[(auc.Setup == name) & (auc.Side == "short")]["AUC validation"].iloc[0]),
        "auc_test_long": float(auc[(auc.Setup == name) & (auc.Side == "long")]["AUC test"].iloc[0]),
        "auc_test_short": float(auc[(auc.Setup == name) & (auc.Side == "short")]["AUC test"].iloc[0]),
        **({k: model_row.iloc[0][k] for k in ["Trades", "Ø net (ticks)", "Profit factor", "t-value",
                                             "Months in profit", "Net total USD/contract", "Max drawdown USD"]}
           if len(model_row) else {}),
        **({f"dir_{k}": dir_chosen[name][k] for k in ["side", "direction_quantile", "timing_quantile"]}
           if name in dir_chosen else {}),
        **({f"dir_{k}": v for k, v in dir_test_df[(dir_test_df["Setup"] == name)
                                                    & (dir_test_df["Strategy"] == "Direction model")].iloc[0][
               ["Trades", "Ø net (ticks)", "Profit factor", "t-value", "Months in profit", "Net total USD/contract"]].items()}
           if len(dir_test_df) and ((dir_test_df["Setup"] == name) & (dir_test_df["Strategy"] == "Direction model")).any() else {}),
        "costs_ticks": COST_PER_TRADE_TICKS, "train_row_stride": TRAIN_ROW_STRIDE, "model_path": str(path.resolve()),
    })
log_df = pd.DataFrame(log_rows)
if LOG_PATH.exists():
    log_df = pd.concat([pd.read_csv(LOG_PATH, sep=";"), log_df], ignore_index=True)
log_df.to_csv(LOG_PATH, sep=";", index=False)
print(f"✓ Logged to {LOG_PATH.resolve()}")

✓ Written: E:\Programierung\Daten\NQ\Candelexport_NQ_M1_MERGED_12.12.2019_26.10.2026_test_signals.csv (353,599 test rows, new columns: ['P_long_15min', 'P_short_15min', 'Signal_15min', 'Target_ticks_15min', 'Stop_ticks_15min', 'P_up_first_15min', 'Trade_15min', 'DirSignal_15min', 'DirTrade_15min', 'P_long_60min', 'P_short_60min', 'Signal_60min', 'Target_ticks_60min', 'Stop_ticks_60min', 'P_up_first_60min', 'Trade_60min', 'DirSignal_60min', 'DirTrade_60min'])
✓ Saved: E:\Programierung\Hitrevo\tradingbrains\trading_brains\trading_brains_main_tft\models_barrier_NQ\barrier_NQ_15min_20261026_212502.joblib
✓ Saved: E:\Programierung\Hitrevo\tradingbrains\trading_brains\trading_brains_main_tft\models_barrier_NQ\barrier_NQ_60min_20261026_212502.joblib
✓ Logged to E:\Programierung\Hitrevo\tradingbrains\trading_brains\trading_brains_main_tft\models_barrier_NQ\barrier_log.csv
